# 🔬 Multimodal Agent (Vision + Clinical Tool Calling) — Step-by-Step Guide

**What is a Multimodal Agent?**  
Multimodal AI models extend beyond text, processing images, audio, and documents alongside natural language prompts.  
When combined with **Agentic Tool Calling**, a Multimodal Agent can:
1. **Perceive Visual Inputs**: Inspect raw images (e.g. medical lab reports, diagrams, photographs).
2. **Extract & Triage Data**: Parse complex unstructured visual tables, identify abnormal values, and categorize conditions.
3. **Execute Domain-Specific Tools**: Automatically invoke backend business logic or clinical lookup functions based on the visual findings.

---

### Multimodal Clinical Agent Architecture

```
┌─────────────────────────────────┐
│        blood_work.png           │  Raw Patient Lab Report (Image)
└────────────────┬────────────────┘
                 │ Base64 Ingestion
                 ▼
┌────────────────────────────────────────────────────────┐
│     Multimodal Vision LLM (Gemini 2.5 / GPT-4o)        │
│                                                        │
│  1. Visual OCR & Numerical Extraction                  │
│     - Total Cholesterol: 238 mg/dL [HIGH]              │
│     - LDL Cholesterol:   162 mg/dL [HIGH]              │
│     - HDL Cholesterol:   36 mg/dL  [LOW]               │
│     - Triglycerides:     188 mg/dL [HIGH]              │
│  2. Clinical Triage Reasoner                           │
│     - Diagnosis: Category "high_cholesterol"           │
│  3. Decision: Call get_diet_recommendation()           │
└────────────────────────┬───────────────────────────────┘
                         │ Tool Call: get_diet_recommendation("high_cholesterol")
                         ▼
┌────────────────────────────────────────────────────────┐
│            get_diet_recommendation() Tool              │
│  Eat: fruits, vegetables, whole grains, lean protein   │
│  Avoid: red meat, fried food, full-fat dairy           │
└────────────────────────┬───────────────────────────────┘
                         │ Tool Observation
                         ▼
┌────────────────────────────────────────────────────────┐
│                 Synthesized Final Report               │
│  - Biomarker Summary Table with High/Low flags         │
│  - Categorization: high_cholesterol                    │
│  - Targeted Dietary Recommendations & Doctor Advice    │
└────────────────────────────────────────────────────────┘
```

**Key Concepts Covered:**
1. **Multimodal Ingestion**: Encoding images into base64 Data URLs and packaging them into LangChain `HumanMessage` structures.
2. **Visual Feature Extraction**: Extracting tabular clinical markers (CBC, Lipids, Metabolic Panel) directly from pixel data.
3. **Condition Classification**: Reasoning over extracted values to classify clinical status (`normal`, `high_cholesterol`, `high_sugar`).
4. **Conditional Tool Dispatch**: Triggering specialized recommendation tools based on visual classification.


---
## Step 1 — Setup & Multimodal Vision Dependencies

We load environment variables and import LangChain components for multimodal messages:
- `HumanMessage`: Accommodates mixed-content payloads (both text and image URLs).
- `base64`: Encodes image files on disk into base64 strings compatible with multimodal vision APIs.


In [5]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

---
## Step 2 — Direct Visual Inspection & Extraction (`blood_work.png`)

Before creating the agent, we perform a direct visual inspection using the vision model.
We encode `blood_work.png` into base64 and prompt the model to extract all lab markers and flag abnormal values.


In [7]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

llm = ChatOpenAI(
    model="gpt-6-luna",
    # temperature=0,
)

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

**Patient:** Rajesh Sharma, 48, male  
**Report date:** May 7, 2026

| Test | Result | Reference range | Flag |
|---|---:|---:|---|
| **CBC** ||||
| Hemoglobin | 15.1 g/dL | 13.5–17.5 g/dL | Within range |
| Hematocrit | 44% | 41–53% | Within range |
| WBC | 6.8 ×10³/µL | 4.5–11.0 ×10³/µL | Within range |
| Platelets | 220 ×10³/µL | 150–400 ×10³/µL | Within range |
| **Lipid panel** ||||
| Total cholesterol | 238 mg/dL | <200 mg/dL | **High** |
| LDL cholesterol | 162 mg/dL | <100 mg/dL | **High** |
| HDL cholesterol | 36 mg/dL | >40 mg/dL | **Low** |
| Triglycerides | 188 mg/dL | <150 mg/dL | **High** |
| **Metabolic panel** ||||
| Fasting glucose | 92 mg/dL | 70–99 mg/dL | Within range |
| HbA1c | 5.3% | <5.7% | Within range |
| Creatinine | 1.0 mg/dL | 0.7–1.3 mg/dL | Within range |
| eGFR | 82 mL/min | >60 mL/min | Within range |

The image shows a **Liver Function** heading, but no liver test results are visible.


---
## Step 3 — Nutrition Knowledge Base & Clinical Tool Definition (`get_diet_recommendation`)

We define the clinical recommendation tool. Given a diagnosed condition (`high_cholesterol`, `high_sugar`, `normal`), it returns evidence-based dietary recommendations.


In [8]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

---
## Step 4 — Clinical Multimodal Agent Construction

We configure the multimodal agent:
- **System Instructions**: Explicitly instruct the model to:
  1. Extract numbers and normal reference ranges from the image.
  2. Categorize the clinical condition into one of: `normal`, `high_cholesterol`, `high_sugar`.
  3. Call `get_diet_recommendation` with the identified category.


In [9]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

---
## Step 5 — End-to-End Multimodal Execution (Visual Triage + Tool Calling + Diet Plan)

We feed both the image and the clinical prompt into the agent.
The agent:
1. Reads `blood_work.png`.
2. Identifies elevated Total Cholesterol (238) and LDL (162).
3. Classifies the condition as `high_cholesterol`.
4. Invokes `get_diet_recommendation(condition="high_cholesterol")`.
5. Synthesizes a clinical summary and tailored diet plan!


In [11]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

def get_text(content):
    if isinstance(content, list):
        return "\n".join(item["text"] for item in content if isinstance(item, dict) and item.get("type") == "text")
    return content

print(get_text(result["messages"][-1].content))

## Blood work summary

**Patient:** Rajesh Sharma, 48, male  
**Date:** May 7, 2026

| Test | Result | Reported reference range | Interpretation |
|---|---:|---:|---|
| Hemoglobin | 15.1 g/dL | 13.5–17.5 | Within range |
| Hematocrit | 44% | 41–53% | Within range |
| WBC | 6.8 ×10³/µL | 4.5–11.0 | Within range |
| Platelets | 220 ×10³/µL | 150–400 | Within range |
| **Total cholesterol** | **238 mg/dL** | **<200** | **High** |
| **LDL cholesterol** | **162 mg/dL** | **<100** | **High** |
| **HDL cholesterol** | **36 mg/dL** | **>40** | **Low** |
| **Triglycerides** | **188 mg/dL** | **<150** | **High** |
| Fasting glucose | 92 mg/dL | 70–99 | Within range |
| HbA1c | 5.3% | <5.7% | Within range |
| Creatinine | 1.0 mg/dL | 0.7–1.3 | Within range |
| eGFR | 82 mL/min | >60 | Within range |

**Category: high_cholesterol.** The glucose and HbA1c are within the stated ranges. The liver-function section is cut off in the image, so its results can’t be assessed.

## Diet plan

The retrieved 